# 00 · 数据准备
按 00 → 05 顺序运行。优先读取本地 config.local.json，否则读取公共 config.json。可设置 SOURCE_ROOT 和 RUN_NAME；实际配置写入 outputs/active_run_config.json，后续 Notebook 自动读取。运行本 Notebook 会重新建立同名结果目录。


In [ ]:
from pathlib import Path
import os, sys
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "prelib").is_dir() and (p / "config.json").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("请在包含 prelib 和 config.json 的项目目录中运行。")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
for name in ["OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(name, "4")
from threadpoolctl import threadpool_limits
thread_limiter = threadpool_limits(limits=4)
import pandas as pd
from IPython.display import display, Image
from prelib.common import (load_config, domain_config, run_dir, shared_dir,
                           initialize_run, set_active_config, write_json, environment_report)


In [ ]:
USE_DEMO = os.environ.get("PRE_DEMO", "0") == "1"
CONFIG_FILE = "config.local.json" if (PROJECT / "config.local.json").is_file() else "config.json"
SOURCE_ROOT = None  # Optional override; prefer config.local.json for machine-specific paths.
RUN_NAME = None
if USE_DEMO:
    if SOURCE_ROOT is not None:
        raise ValueError("Do not combine USE_DEMO with a real-data SOURCE_ROOT override.")
    from prelib.demo import create_demo_source
    create_demo_source()
    CONFIG_FILE = "config_demo.json"
cfg = load_config(PROJECT / CONFIG_FILE)
if SOURCE_ROOT is not None:
    cfg["source_root"] = str(SOURCE_ROOT)
if RUN_NAME is not None:
    cfg["run_name"] = RUN_NAME
from prelib.preflight import check_input_files
check_input_files(cfg)
initialize_run(cfg)
ACTIVE_CONFIG = PROJECT / "outputs/active_run_config.json"
write_json(ACTIVE_CONFIG, cfg)
set_active_config(str(ACTIVE_CONFIG))
write_json(shared_dir(cfg) / "models/config_used.json", cfg)
write_json(shared_dir(cfg) / "models/environment.json", environment_report(cfg))
display(pd.DataFrame(cfg["domains"]))


严格日期顺序定义有向引用边。向量通过专利 ID 对齐；领域网络保留孤点和属性未知节点。有效终点是请求日期、来源覆盖日期、元数据最大日期的最小值。

In [ ]:
from prelib.prepare import prepare_data
summary = prepare_data(cfg)
display(pd.DataFrame([summary]))
display(pd.read_csv(shared_dir(cfg) / "tables/domain_sizes.csv"))
